In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error 

**Importing dataset**

In [23]:
dataset_train = pd.read_csv("train.csv")
dataset_test = pd.read_csv("test.csv")

**From EDA File - dividing columns**

In [24]:
binary_cols = ["internet_access"]

ordinal_cols = ["sleep_quality", "facility_rating", "exam_difficulty"]

ordinal_categories = [
    ["poor", "average", "good"],
    ["low", "medium", "high"],
    ["easy", "moderate", "hard"]
]

onehot_cols = ["gender", "course", "study_method"]

num_cols = ["age", "study_hours", "class_attendance", "sleep_hours"]

**Feature Engineering**

In [25]:
def add_engineered_features(df):
    df_temp = df.copy()
    # Sine features
    df_temp['_study_hours_sin'] = np.sin(2 * np.pi * df_temp['study_hours'] / 12).astype('float32')
    df_temp['_class_attendance_sin'] = np.sin(2 * np.pi * df_temp['class_attendance'] / 12).astype('float32')

    # for col in num_cols:
    #     if col in df_temp.columns:
    #         df_temp[f'log_{col}'] = np.log1p(df_temp[col])
    #         df_temp[f'{col}_sq'] = df_temp[col] ** 2

    # for col in dataset_train.select_dtypes(include=['object','category']).columns.tolist():

    #     cat_series = df_temp[col].astype(str)  
    #     freq_map = cat_series.value_counts().to_dict()        
    #     df_temp[f"{col}_freq"] = cat_series.map(freq_map).fillna(0).astype(int)
        
    # # Linear combo feature
    # df_temp['feature_formula'] = (
    #         5.9051154511950499 * df_temp['study_hours'] +
    #         0.34540967058057986 * df_temp['class_attendance'] +
    #         1.423461171860262 * df_temp['sleep_hours'] + 4.7819
    # )


    return df_temp

In [26]:
dataset_train = add_engineered_features(dataset_train)
dataset_train.columns

Index(['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance',
       'internet_access', 'sleep_hours', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty', 'exam_score', '_study_hours_sin',
       '_class_attendance_sin'],
      dtype='object')

In [27]:
numeric_col = ["age", "study_hours", "class_attendance", "sleep_hours", "_study_hours_sin",
       "_class_attendance_sin",]

In [17]:
# BINARY ENCODING 
binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["no", "yes"]]))
])

# ORDINARY ENCODING
ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=ordinal_categories))
])

# ONE-HOT ENCODING
onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# NUMERIC SCALING
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# PREPROCESSING
preprocessor = ColumnTransformer(
    transformers=[
        ("bin", binary_transformer, binary_cols),
        ("ord", ordinal_transformer, ordinal_cols),
        ("oh", onehot_transformer, onehot_cols),
        ("num", numeric_transformer, num_cols)
    ],
    remainder="drop"
)

#  XGB
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", xgb_model)
])

# LGB
lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", lgb_model)
])


In [ ]:
X = dataset_train.drop(columns=["id", "exam_score"])
y = dataset_train["exam_score"]

X_test = dataset_test.drop(columns=["id"])

kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f"Fold {fold+1}")

#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#     lgb_pipeline.fit(X_train, y_train)

#     # OOF predictions
#     oof_preds[val_idx] = lgb_pipeline.predict(X_val)

#     # test predictions (averaged later)
#     test_preds += lgb_pipeline.predict(X_test) / kf.n_splits

def train_model_cv(model, X, y, X_test, kf):
    oof = np.zeros(len(X))
    test = np.zeros(len(X_test))

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        m = clone(model)
        m.fit(X_train, y_train)

        oof[val_idx] = m.predict(X_val)
        test += m.predict(X_test) / kf.n_splits

    return oof, test


models = [
    lgb_pipeline,
    xgb_pipeline,
]

In [ ]:
oof_matrix = []
test_matrix = []

for model in models:
    oof, test = train_model_cv(model, X, y, X_test, kf)

    oof_matrix.append(oof)
    test_matrix.append(test)

X_meta = np.column_stack(oof_matrix)
X_test_meta = np.column_stack(test_matrix)

meta_model = Ridge()
meta_model.fit(X_meta, y)

final_pred = meta_model.predict(X_test_meta)

In [ ]:
rmse = root_mean_squared_error(y, meta_model.predict(X_meta))
print("OOF RMSE:", rmse)

OOF RMSE: 8.756784347866574
